# 🌍 Küresel CO₂ Emisyonları Analizi — Türkiye Odaklı

**Veri:** [Our World in Data — CO₂ and Greenhouse Gas Emissions](https://github.com/owid/co2-data) (1750–2024, 200+ ülke)

Tüm grafikler **interaktif** (Plotly): üzerine gelince değerleri gösterir, sürükleyerek yakınlaştırabilir, lejanttaki bir seriye tıklayarak gizleyebilirsin.

### Sorular
1. Küresel CO₂ emisyonları nasıl değişti, hangi kaynaklardan geliyor?
2. Kim ne kadar salıyor — toplamda ve kişi başına?
3. Türkiye'nin emisyonları nasıl gelişti?
4. Ekonomik büyüme ile emisyon arasındaki bağ kopuyor mu?
5. Ülkeler emisyon profillerine göre nasıl gruplanıyor? (K-Means)
6. Mevcut eğilim sürerse Türkiye 2030'da nerede olur?

> Birimler: `co2` milyon ton (Mt), `co2_per_capita` ton/kişi.

## 1. Kurulum ve veri

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

# Renk-körü dostu sabit palet
MAVI, TURUNCU, SU_YESILI, SARI, PEMBE, YESIL, MOR, KIRMIZI = (
    "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948")
PALET = [MAVI, TURUNCU, SU_YESILI, SARI, PEMBE, YESIL, MOR, KIRMIZI]
GRI = "#52514e"

pio.renderers.default = "plotly_mimetype+notebook_connected"  # JupyterLab + nbviewer
px.defaults.template = "plotly_white"
px.defaults.color_discrete_sequence = PALET
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Veri ilk çalıştırmada indirilir, sonra yerel kopyadan okunur
URL = "https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv"
DOSYA = Path("owid-co2-data.csv")
if not DOSYA.exists():
    pd.read_csv(URL).to_csv(DOSYA, index=False)
df = pd.read_csv(DOSYA)

# Bölge/kıta toplamlarını ayır: gerçek ülkelerin ISO kodu var ve "OWID" ile başlamıyor
ulkeler = df[df.iso_code.notna() & ~df.iso_code.astype(str).str.startswith("OWID")].copy()
dunya = df[df.country == "World"].set_index("year")
tr = df[df.country == "Turkey"].set_index("year")
SON_YIL = int(ulkeler.year.max())
GSYIH_SON = int(df.dropna(subset=["gdp"]).year.max())   # GSYİH serisi kaynakta daha erken bitiyor

print(f"{df.shape[0]:,} satır, {df.shape[1]} sütun | {ulkeler.country.nunique()} ülke | {df.year.min()}–{SON_YIL}")

50,411 satır, 79 sütun | 218 ülke | 1750–2024


## 2. Küresel tablo: emisyonlar nereden geliyor?

In [2]:
kaynaklar = {"coal_co2": "Kömür", "oil_co2": "Petrol", "gas_co2": "Doğalgaz",
             "cement_co2": "Çimento", "flaring_co2": "Gaz yakma", "other_industry_co2": "Diğer"}
k = (dunya.loc[1850:, list(kaynaklar)].fillna(0).rename(columns=kaynaklar)
     .reset_index().melt("year", var_name="Kaynak", value_name="Mt"))

fig = px.area(k, x="year", y="Mt", color="Kaynak",
              title="Küresel fosil CO₂ emisyonları — kaynağa göre (Mt)",
              labels={"year": "", "Mt": "Milyon ton CO₂"})
fig.update_traces(line_width=0.5, hovertemplate="%{y:,.0f} Mt")
fig.update_layout(hovermode="x unified", legend_title_text="")
fig.show()

print(f"{SON_YIL}: {dunya.loc[SON_YIL,'co2']:,.0f} Mt  |  1990'a göre %{(dunya.loc[SON_YIL,'co2']/dunya.loc[1990,'co2']-1)*100:.0f} artış  |  "
      f"kömür payı %{dunya.loc[SON_YIL,'coal_co2']/dunya.loc[SON_YIL,'co2']*100:.0f}")

2024: 38,599 Mt  |  1990'a göre %70 artış  |  kömür payı %41


## 3. Dünya haritası — kaydırıcıyla yıllar arasında gez

Alttaki ▶ düğmesiyle animasyonu oynatabilir ya da kaydırıcıyı sürükleyebilirsin.

In [3]:
harita = ulkeler[ulkeler.year.between(1990, SON_YIL)].dropna(subset=["co2_per_capita"])
fig = px.choropleth(harita, locations="iso_code", color="co2_per_capita", hover_name="country",
                    animation_frame="year", range_color=(0, 20), color_continuous_scale="Blues",
                    hover_data={"iso_code": False, "year": False, "co2_per_capita": ":.1f", "co2": ":,.0f"},
                    labels={"co2_per_capita": "ton/kişi", "co2": "Toplam (Mt)"},
                    title="Kişi başı CO₂ emisyonu (ton/kişi, 20+ aynı renk)")
fig.update_geos(showframe=False, showcoastlines=False, projection_type="natural earth")
fig.update_layout(height=560, margin=dict(l=0, r=0, t=50, b=0))
fig.layout.sliders[0].active = len(fig.frames) - 1          # son yılı göstererek başla
fig.update_traces(z=fig.frames[-1].data[0].z, locations=fig.frames[-1].data[0].locations,
                  hovertext=fig.frames[-1].data[0].hovertext, customdata=fig.frames[-1].data[0].customdata)
fig.show()

## 4. En büyük emitörler

Sağ üstteki menüden **toplam / kişi başı / kümülatif** arasında geçiş yap. Türkiye turuncu.

In [4]:
son = ulkeler[ulkeler.year == SON_YIL]
son_buyuk = son[son.population > 1_000_000]   # kişi başı listeyi minik ülkeler domine etmesin

olcutler = [("co2", "Toplam (Mt)", son), ("co2_per_capita", "Kişi başı (ton)", son_buyuk),
            ("cumulative_co2", "Kümülatif 1750'den beri (Mt)", son)]
fig = go.Figure()
for i, (kol, ad, veri) in enumerate(olcutler):
    top = veri.nlargest(15, kol).iloc[::-1]
    fig.add_bar(x=top[kol], y=top.country, orientation="h", name=ad, visible=(i == 0),
                marker_color=[TURUNCU if c == "Turkey" else MAVI for c in top.country],
                hovertemplate="%{y}: %{x:,.1f}<extra></extra>")
fig.update_layout(
    title=f"En çok emisyon yapan 15 ülke ({SON_YIL})", height=520, showlegend=False,
    updatemenus=[dict(x=1, xanchor="right", y=1.12, buttons=[
        dict(label=ad, method="update", args=[{"visible": [j == i for j in range(3)]}])
        for i, (_, ad, _) in enumerate(olcutler)])])
fig.show()

sira = int(son.co2.rank(ascending=False)[son.country == "Turkey"].iloc[0])
print(f"Türkiye toplam emisyonda {sira}. sırada, küresel payı %{tr.loc[SON_YIL,'share_global_co2']:.2f}")

Türkiye toplam emisyonda 12. sırada, küresel payı %1.33


## 5. Türkiye

In [5]:
fig = make_subplots(1, 2, subplot_titles=("Türkiye CO₂ — kaynağa göre (Mt)", "Kişi başı emisyon (ton/kişi)"),
                    horizontal_spacing=0.08)
t = tr.loc[1960:]
for (kol, ad), renk in zip({"coal_co2": "Kömür", "oil_co2": "Petrol", "gas_co2": "Doğalgaz",
                             "cement_co2": "Çimento"}.items(), PALET):
    fig.add_scatter(x=t.index, y=t[kol], name=ad, stackgroup="1", line=dict(width=0.5, color=renk),
                    hovertemplate=ad + ": %{y:,.0f} Mt<extra></extra>", row=1, col=1)

for ad, ulke, renk, kalin in [("Türkiye", "Turkey", TURUNCU, 3.5), ("Dünya", "World", GRI, 2),
                              ("AB (27)", "European Union (27)", MOR, 2), ("Çin", "China", KIRMIZI, 2)]:
    s = df[(df.country == ulke) & (df.year >= 1960)]
    fig.add_scatter(x=s.year, y=s.co2_per_capita, name=ad, line=dict(color=renk, width=kalin),
                    hovertemplate=ad + ": %{y:.2f} t<extra></extra>", row=1, col=2)
fig.update_layout(height=470, hovermode="x unified", legend=dict(orientation="h", y=-0.12))
fig.show()

ozet = pd.DataFrame({y: {"Toplam (Mt)": tr.loc[y, "co2"], "Kişi başı (t)": tr.loc[y, "co2_per_capita"],
                         "Kömür payı (%)": tr.loc[y, "coal_co2"] / tr.loc[y, "co2"] * 100,
                         "Nüfus (milyon)": tr.loc[y, "population"] / 1e6}
                     for y in [1990, 2000, 2010, SON_YIL]}).T
ozet

,Toplam (Mt),Kişi başı (t),Kömür payı (%),Nüfus (milyon)
1990,154.57,2.76,45.54,56.02
2000,232.93,3.56,42.39,65.43
2010,318.71,4.34,40.75,73.35
2024,513.03,5.87,40.79,87.47


## 6. Büyüme ve emisyon: bağ kopuyor mu?

GSYİH ve CO₂ 1990 = 100 olarak endekslendi. Menüden ülke seç. Mavi çizgi turuncunun üzerinde kalıp ayrışıyorsa ekonomi emisyondan daha hızlı büyüyor demektir.

In [6]:
secim = {"Türkiye": "Turkey", "Almanya": "Germany", "ABD": "United States",
         "Birleşik Krallık": "United Kingdom", "Çin": "China", "Hindistan": "India", "Polonya": "Poland"}
fig = go.Figure()
for i, (ad, ulke) in enumerate(secim.items()):
    s = df[(df.country == ulke) & df.year.between(1990, GSYIH_SON)].set_index("year")
    for kol, etiket, renk in [("gdp", "GSYİH", MAVI), ("co2", "CO₂", TURUNCU)]:
        fig.add_scatter(x=s.index, y=s[kol] / s.loc[1990, kol] * 100, name=etiket, visible=(i == 0),
                        line=dict(color=renk, width=3), hovertemplate=etiket + ": %{y:.0f}<extra></extra>")
fig.add_hline(y=100, line_dash="dot", line_color=GRI)
n = len(secim)
fig.update_layout(
    title=f"GSYİH ve CO₂ endeksi (1990 = 100), 1990–{GSYIH_SON}", height=460, hovermode="x unified",
    updatemenus=[dict(x=1, xanchor="right", y=1.15, buttons=[
        dict(label=ad, method="update", args=[{"visible": [j // 2 == i for j in range(2 * n)]}])
        for i, ad in enumerate(secim)])])
fig.show()

### Zenginleştikçe ülkeler nasıl hareket etti? (animasyon)
Balon büyüklüğü = nüfus. ▶ ile oynat, bir balonun üzerine gelince ülkeyi gör.

In [7]:
b = ulkeler[ulkeler.year.between(1990, GSYIH_SON) & (ulkeler.population > 5_000_000)] \
    .dropna(subset=["gdp", "co2_per_capita"]).copy()
b["Kişi başı GSYİH ($)"] = b.gdp / b.population
b["Grup"] = np.where(b.country == "Turkey", "Türkiye", "Diğer ülkeler")

fig = px.scatter(b, x="Kişi başı GSYİH ($)", y="co2_per_capita", size="population", color="Grup",
                 hover_name="country", animation_frame="year", animation_group="country",
                 log_x=True, log_y=True, size_max=55, range_x=[400, 120_000], range_y=[0.03, 60],
                 color_discrete_map={"Türkiye": TURUNCU, "Diğer ülkeler": MAVI},
                 labels={"co2_per_capita": "ton CO₂ / kişi"},
                 title="Kişi başı GSYİH vs kişi başı CO₂ (log-log)")
fig.update_traces(marker=dict(line=dict(width=1, color="white"), opacity=0.8))
fig.update_layout(height=560, legend_title_text="")
fig.show()

## 7. Makine öğrenmesi: K-Means ile ülke grupları

Özellikler: kişi başı CO₂, kişi başı enerji, karbon yoğunluğu (enerji başına CO₂), kömür payı ve kişi başı GSYİH (log). Veriyi standartlaştırıp küme sayısını **siluet skoru** ile seçiyoruz.

In [8]:
k = ulkeler[(ulkeler.year == GSYIH_SON) & (ulkeler.population > 1_000_000)].copy()
k["gdp_pc"] = k.gdp / k.population
k["log_gdp_pc"] = np.log(k.gdp_pc)
k["komur_payi"] = k.coal_co2.fillna(0) / k.co2 * 100
ozellikler = ["co2_per_capita", "energy_per_capita", "co2_per_unit_energy", "komur_payi", "log_gdp_pc"]
k = k.dropna(subset=ozellikler)
X = StandardScaler().fit_transform(k[ozellikler])

skor = {n: silhouette_score(X, KMeans(n, n_init=20, random_state=42).fit_predict(X)) for n in range(2, 8)}
K = min(max(skor, key=skor.get), 4)        # yorumlanabilirlik için en fazla 4 grup
k["kume"] = KMeans(K, n_init=20, random_state=42).fit_predict(X)
print("Siluet skorları:", {n: round(s, 3) for n, s in skor.items()}, f"→ k = {K}")

# Kümelere profil özelliklerine göre isim ver
med = k.groupby("kume")[ozellikler].median()
ad = {med.co2_per_capita.idxmax(): "Çok yüksek – enerji yoğun"}
ad[med.drop(list(ad)).co2_per_capita.idxmin()] = "Düşük emisyon – gelişmekte"
kalan = med.drop(list(ad))
if len(kalan) == 2:
    ad[kalan.komur_payi.idxmax()] = "Kömür ağırlıklı"
    ad[kalan.komur_payi.idxmin()] = "Sanayileşmiş – orta/yüksek gelir"
elif len(kalan) == 1:
    ad[kalan.index[0]] = "Orta/yüksek gelir"
k["Grup"] = k.kume.map(ad)
sira = list(k.groupby("Grup").co2_per_capita.median().sort_values().index)

fig = px.scatter(k, x="gdp_pc", y="co2_per_capita", color="Grup", hover_name="country",
                 category_orders={"Grup": sira}, log_x=True, log_y=True,
                 hover_data={"komur_payi": ":.0f", "gdp_pc": ":,.0f", "co2_per_capita": ":.1f", "Grup": False},
                 labels={"gdp_pc": "Kişi başı GSYİH ($)", "co2_per_capita": "ton CO₂ / kişi", "komur_payi": "Kömür payı %"},
                 title=f"K-Means ile {len(k)} ülke, {K} grup ({GSYIH_SON})")
fig.update_traces(marker=dict(size=11, line=dict(width=1.5, color="white")))
t_ = k[k.country == "Turkey"].iloc[0]
fig.add_annotation(x=np.log10(t_.gdp_pc), y=np.log10(t_.co2_per_capita), text="<b>Türkiye</b>",
                   showarrow=True, arrowhead=0, ax=40, ay=-35)
fig.update_layout(height=560)
fig.show()

# Türkiye'ye profil olarak en çok benzeyen ülkeler
i = np.where(k.country.values == "Turkey")[0][0]
benzer = k.assign(mesafe=np.linalg.norm(X - X[i], axis=1)).nsmallest(6, "mesafe").iloc[1:]
print("Türkiye'nin grubu:", t_.Grup, "| En benzer 5 ülke:", ", ".join(benzer.country))
k.groupby("Grup")[["co2_per_capita", "komur_payi", "gdp_pc"]].median().loc[sira] \
 .assign(ulke_sayisi=k.Grup.value_counts())

Siluet skorları: {2: 0.331, 3: 0.317, 4: 0.359, 5: 0.366, 6: 0.324, 7: 0.298} → k = 4


Türkiye'nin grubu: Sanayileşmiş – orta/yüksek gelir | En benzer 5 ülke: Mauritius, North Macedonia, Slovakia, Malaysia, Thailand


,co2_per_capita,komur_payi,gdp_pc,ulke_sayisi
Grup,,,,
Düşük emisyon – gelişmekte,0.59,0.09,"3,786.63",61
Kömür ağırlıklı,3.13,55.35,"10,252.08",26
Sanayileşmiş – orta/yüksek gelir,4.63,12.83,"26,789.34",56
Çok yüksek – enerji yoğun,20.48,0.18,"52,197.42",10


## 8. Türkiye 2030 projeksiyonu

Son 15 yılın eğilimine doğrusal regresyon + artıkların **bootstrap**'i ile %90 belirsizlik aralığı.

> ⚠️ Bu bir politika senaryosu değil; yalnızca "mevcut eğilim sürerse" sorusunun istatistiksel cevabı.

In [9]:
p = tr.loc[SON_YIL - 14:SON_YIL, "co2"]
yil = p.index.values.reshape(-1, 1)
lr = LinearRegression().fit(yil, p.values)
gelecek = np.arange(SON_YIL, 2031).reshape(-1, 1)
tahmin = lr.predict(gelecek)

rng = np.random.default_rng(42)
artik = p.values - lr.predict(yil)
boot = np.array([LinearRegression().fit(yil, lr.predict(yil) + rng.choice(artik, len(artik))).predict(gelecek)
                 for _ in range(1000)])
alt, ust = np.percentile(boot, [5, 95], axis=0)
x = gelecek.ravel()

fig = go.Figure()
fig.add_scatter(x=np.r_[x, x[::-1]], y=np.r_[ust, alt[::-1]], fill="toself", fillcolor="rgba(235,104,52,0.18)",
                line_width=0, name="%90 aralık", hoverinfo="skip")
fig.add_scatter(x=tr.loc[1990:].index, y=tr.loc[1990:, "co2"], name="Gerçekleşen",
                line=dict(color=MAVI, width=3), hovertemplate="%{x}: %{y:.0f} Mt<extra></extra>")
fig.add_scatter(x=x, y=tahmin, name="Eğilim", line=dict(color=TURUNCU, width=3, dash="dash"),
                hovertemplate="%{x}: %{y:.0f} Mt<extra></extra>")
fig.update_layout(title="Türkiye CO₂ emisyonları ve 2030 eğilim projeksiyonu (Mt)", height=450,
                  hovermode="x unified", legend=dict(x=0.01, y=0.99))
fig.show()

print(f"Eğim: yılda +{lr.coef_[0]:.1f} Mt (R² = {lr.score(yil, p.values):.2f})  |  "
      f"2030: ~{tahmin[-1]:.0f} Mt (%90: {alt[-1]:.0f}–{ust[-1]:.0f})  |  {SON_YIL}'e göre %{(tahmin[-1]/tr.loc[SON_YIL,'co2']-1)*100:+.0f}")

Eğim: yılda +11.7 Mt (R² = 0.92)  |  2030: ~559 Mt (%90: 540–579)  |  2024'e göre %+9


## 9. Sonuçlar

| # | Soru | Bulgu |
|---|---|---|
| 1 | Küresel eğilim | 2024'te küresel fosil CO₂ **~38,6 milyar ton**; 1990'a göre **%70 artış**. Kömür hâlâ en büyük kaynak (**%41**). |
| 2 | En büyük emitörler | Toplamda Çin, ABD, Hindistan önde; kişi başında Körfez ülkeleri zirvede. Kümülatifte ABD ilk sırada. |
| 3 | Türkiye | Dünyada **12. sırada** (küresel pay **%1,33**). 1990'dan beri emisyon **3,3 kat**, nüfus 1,56 kat arttı. Kişi başı **5,9 ton** ile dünya ortalamasını (4,7) ve AB'yi (5,4) geçti. Kömür payı 35 yıldır **~%41**'de sabit. |
| 4 | Decoupling | Almanya ve İngiltere'de **mutlak ayrışma** (GSYİH ↑, CO₂ ↓). Türkiye'de **göreli ayrışma**: 1990–2022'de GSYİH ~4,9 kat, CO₂ ~2,9 kat büyüdü. |
| 5 | Kümeleme | 153 ülke 4 profile ayrıldı; Türkiye **"Sanayileşmiş – orta/yüksek gelir"** grubunda. |
| 6 | 2030 | Eğilim sürerse **~559 Mt** (%90 aralık 540–579) — 2024'e göre ~%9 artış. |

**Genel çıkarım:** Türkiye büyümeyi emisyondan kısmen ayırabilmiş ama mutlak düşüşe geçmemiş; kömür bağımlılığının azalmaması kişi başı emisyonun AB'yi geçmesinin temel nedenlerinden biri.